# ML forcing scatter per site

predicted τx τy QT vs bulk formula truth, broken down by site. shows the MLP's per site skill instead of just aggregate. for each target, 6 panels, one per location.

In [ ]:
using JLD2
using Plots
using Statistics

gr()
default(linewidth=1.5)

const ROOT = normpath(joinpath(@__DIR__, ".."))
const SITES = ["lat30lon-50", "lat-25lon-10", "lat-45lon80", "lat0lon-140", "lat30lon-150", "lat40lon-25"]

v2_res = JLD2.load(joinpath(ROOT, "data/generated/era5_forcing_results_v2.jld2"))
Yte = v2_res["Yte"]
Yte_mlp = v2_res["Yte_mlp"]
Yte_lin = v2_res["Yte_lin"]
target_names = v2_res["target_names"]
test_idx = v2_res["test_idx"]

# figure out which rows belong to which site
ds = JLD2.load(joinpath(ROOT, "data/generated/era5_forcing_dataset_v2.jld2"))
source_file = ds["source_file"][test_idx]

In [ ]:
function scatter_by_site(j, target_label)
    plots = Plots.Plot[]
    for site in SITES
        mask = occursin.(site, source_file)
        any(mask) || continue
        t = Yte[mask, j]
        mlp = Yte_mlp[mask, j]
        lo, hi = minimum(t), maximum(t)
        plt = scatter(t, mlp, color=:red, markersize=1.0, alpha=0.4,
                      xlabel="truth", ylabel="v2 MLP",
                      title=site, titlefontsize=9, legend=false)
        plot!(plt, [lo,hi], [lo,hi], color=:black, linestyle=:dash, lw=1)
        push!(plots, plt)
    end
    plot(plots..., layout=(3,2), size=(1100,1000), plot_title=target_label)
end

In [ ]:
scatter_by_site(1, "τx — v2 MLP per site")

In [ ]:
scatter_by_site(2, "τy — v2 MLP per site")

In [ ]:
scatter_by_site(3, "QT — v2 MLP per site")

## per site RMSE table

In [ ]:
using Printf
println(rpad("site",14), " | ", rpad("τx RMSE",10), rpad("τy RMSE",10), "QT RMSE")
println("-"^50)
for site in SITES
    mask = occursin.(site, source_file)
    any(mask) || continue
    r_tx = sqrt(mean((Yte_mlp[mask,1] .- Yte[mask,1]).^2))
    r_ty = sqrt(mean((Yte_mlp[mask,2] .- Yte[mask,2]).^2))
    r_qt = sqrt(mean((Yte_mlp[mask,3] .- Yte[mask,3]).^2))
    @printf("%-14s | %-10.4f %-10.4f %.1f\n", site, r_tx, r_ty, r_qt)
end